In [ ]:
import pickle
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from common.utils import collect_df  # only for isna in xlabel
from common.consts import res_colors
def plot_mfpt_corr(df, prop_col, wt_label='WT', group_col='residue_idx',
                   annotate=False, th=None, log_prop=True):
    wt_mfpt = float(df.loc[wt_label, 'mfpt'])
    wt_prop = df.loc[wt_label, prop_col]

    x_all = df[prop_col].to_numpy()
    y_all = np.log(df['mfpt'].to_numpy())
    names = df.index.to_numpy()

    x_map = pd.Series(x_all, index=df.index)
    y_map = pd.Series(y_all, index=df.index)

    pearson, p = stats.pearsonr(x_all, y_all)
    spearman, _ = stats.spearmanr(x_all, y_all)
    print(f"p: {p}")

    for k in sorted(int(k) for k in df[group_col].dropna().unique()):
        sub = df[df[group_col] == k]
        plt.scatter(
            x_map.loc[sub.index].to_numpy(),
            y_map.loc[sub.index].to_numpy(),
            label=f"Residue {k}", s=35, alpha=0.9, c=res_colors.get(k)
        )

    plt.scatter(wt_prop, y_map.loc[wt_label], s=120, marker='*', edgecolors='k', linewidths=1.5, label=wt_label)

    if annotate:
        for xv, yv, name in zip(x_all, y_all, names):
            plt.text(xv, yv + 0.05, name, fontsize=7, ha='center', va='bottom')

    xlabel = (f"Mut {prop_col}" if log_prop and not pd.isna(wt_prop) else prop_col)
    plt.xlabel(xlabel)
    plt.ylabel('log(MFPT mutation)')
    t = f" | Th={th:.3g}" if th is not None else ""
    plt.title(f"MFPT vs {prop_col}{t} | Pear: {pearson:.3g}, Spear: {spearman:.3g}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return pearson, spearman



with open("../data/mfpt-pace=25000.pkl", "rb") as f:
    all_mfpt = pickle.load(f)

thresholds = np.array(list(all_mfpt['chignolin'].keys()))
df = collect_df(False, all_mfpt, thresholds[1])
plot_mfpt_corr(df, 'eigenvalue', annotate=True)
plot_mfpt_corr(df, 'abs_dvar_folded', annotate=True)

p: 0.00017687917436862033


NameError: name 'res_colors' is not defined